In [358]:
from graphviz import Digraph
import math
import random
import matplotlib.pyplot as plt
%matplotlib inline

In [233]:
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v.prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root, format='svg', rankdir='LR'):
    """
    format: png | svg | ...
    rankdir: TB (top to bottom graph) | LR (left to right)
    """
    assert rankdir in ['LR', 'TB']
    nodes, edges = trace(root)
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir}) #, node_attr={'rankdir': 'TB'})
    
    for n in nodes:
        dot.node(name=str(id(n)), label = "{ data %.4f | grad %.4f }" % (n.data, n.grad), shape='record')
        if n.op:
            dot.node(name=str(id(n)) + n.op, label=n.op)
            dot.edge(str(id(n)) + n.op, str(id(n)))
    
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2.op)
    
    return dot

In [332]:
class Value:
    def __init__(self, n, children=(), op=''):
        self.data = n
        self.grad = 0
        self.op = op
        self.prev = set(children)
        self._backward = lambda: None

    def __repr__(self):
        return f'Value(data={self.data}, grad={self.grad})'

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out =  Value(self.data + other.data, (self, other), '+')

        def backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f'**{other}')
        
        def backward():
            self.grad += other * (self.data**(other-1)) * out.grad

        out._backward = backward
        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def backward():
            self.grad += (self.data > 0) * out.grad

        out._backward = backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(x) - math.exp(-x)) / (math.exp(x) + math.exp(-x))
        out = Value(t, (self,), 'tanh')

        def backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = backward
        return out

    def __neg__(self):
        return Value(-self.data)

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return self - other

    def __rmul__(self, other):
        return other * self

    def __truediv__(self, other):
        return self * other**-1

    def __rtruediv__(self, other):
        return other * self**-1

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            v._backward()




In [408]:
class Module:
    def zero_grad(self):
        for p in self.parameters:
            p.grad = 0

    def parameters(self):
        return []

        
class Neuron(Module):
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def parameters(self):
        return self.w + [self.b]

    def __call__(self, x):
        act = sum([wi*xi for wi, xi in zip(self.w, x)]) + self.b
        out = act.tanh()
        return out

    def __repr__(self):
        return f'Neuron({len(self.w)})'

class Layer(Module):
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out)==1 else out

    def __repr__(self):
        return f"Layer of [{',\n\t'.join(str(n) for n in self.neurons)}]"

class MLP(Module):
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def parameters(self):
        return [p for layer in self.layers  for p in layer.parameters()]

    def __repr__(self):
        return f'MLP of [\n{', \n\t\n'.join(str(layer) for layer in self.layers)}]'

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

In [409]:
xs = [
    [2.3, 5.6, 7.8, -1.1],
    [3.1, 4.3, -2.2, -0.5],
    [6.0, -0.4, 2.1, -2.2],
    [1.1, -3.6, 9.8, 1.0]
]
ys = [1.0, 0.0, -0.8, 0.6]

In [410]:
n = MLP(4, [5,5, 5,1])
n


MLP of [
Layer of [Neuron(4),
	Neuron(4),
	Neuron(4),
	Neuron(4),
	Neuron(4)], 
	
Layer of [Neuron(5),
	Neuron(5),
	Neuron(5),
	Neuron(5),
	Neuron(5)], 
	
Layer of [Neuron(5),
	Neuron(5),
	Neuron(5),
	Neuron(5),
	Neuron(5)], 
	
Layer of [Neuron(5)]]

In [389]:
lr = 0.01
epoch = 100

for k in range(epoch):

    #forward pass
    ypreds = [n(x) for x in xs]
    loss = sum([(ygt - ypred)**2 for ygt, ypred in zip(ys, ypreds)])
    print('loss = ', loss)

    #zero grad
    for p in n.parameters():
        p.grad = 0
        
    #backward pass
    loss.backward()

    #update
    for p in n.parameters():
        p.data += -lr * p.grad


loss =  Value(data=7.331468616930206, grad=0)
loss =  Value(data=3.8262548800462786, grad=0)
loss =  Value(data=3.6364904471474864, grad=0)
loss =  Value(data=3.4508568911002757, grad=0)
loss =  Value(data=3.252945542452116, grad=0)
loss =  Value(data=2.9507674418146017, grad=0)
loss =  Value(data=2.4270210326575468, grad=0)
loss =  Value(data=1.5339931023015596, grad=0)
loss =  Value(data=0.4018513330310662, grad=0)
loss =  Value(data=0.1178891232099482, grad=0)
loss =  Value(data=0.07286317223340356, grad=0)
loss =  Value(data=0.05294517820310754, grad=0)
loss =  Value(data=0.041578174979187296, grad=0)
loss =  Value(data=0.034276645554296686, grad=0)
loss =  Value(data=0.029252834523801317, grad=0)
loss =  Value(data=0.025633797544615634, grad=0)
loss =  Value(data=0.02293721337149018, grad=0)
loss =  Value(data=0.02087383380842028, grad=0)
loss =  Value(data=0.019259882417221503, grad=0)
loss =  Value(data=0.017973458438694363, grad=0)
loss =  Value(data=0.016930937046514025, grad=

In [390]:
ypreds = [n(x) for x in xs]
ypreds

[Value(data=0.9204666460505081, grad=0),
 Value(data=0.0017617884913626568, grad=0),
 Value(data=-0.8010013876831763, grad=0),
 Value(data=0.6121996703140936, grad=0)]